In [6]:
# import cell
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from prettytable import PrettyTable
from utils import stats, exploration_plots
import preprocessing
importlib.reload(preprocessing)
from preprocessing import preprocessing_dev, preprocessing_eval
import sys
import importlib
from sklearn.preprocessing import StandardScaler
import re
#importlib.reload(preprocessing)


In [7]:
df = pd.read_csv("development.csv", header="infer")
evaluate_df = pd.read_csv("evaluation.csv", header="infer")

# df['id'].to_csv("development_ids.csv", index=False)

In [8]:
df = preprocessing_dev(df)
evaluate_df = preprocessing_eval(df,evaluate_df)
y = df["price"]
df.drop(columns=["price","square_feet_quantile"], inplace=True)
evaluate_df.drop(columns = "square_feet_quantile", inplace = True)
# df = df.fillna(1)

square feet quantile: 0        5
1        2
2        1
3        4
4        4
        ..
79584    9
79585    8
79586    0
79587    0
79588    0
Name: square_feet_quantile, Length: 79566, dtype: int64


In [9]:
missing_per_feature = df.isnull().sum()
print(missing_per_feature)

category              0
bathrooms             0
bedrooms              0
fee                   0
has_photo             0
pets_allowed          0
square_feet           0
latitude              0
longitude             0
source                0
time                  0
id                    0
max_from_body         0
min_from_body         0
std_from_body         0
mean_from_body        0
max_from_title        0
min_from_title        0
std_from_title        0
mean_from_title       0
AC                    0
Alarm                 0
Basketball            0
Cable or Satellite    0
Clubhouse             0
Dishwasher            0
Doorman               0
Elevator              0
Fireplace             0
Garbage Disposal      0
Gated                 0
Golf                  0
Gym                   0
Hot Tub               0
Internet Access       0
Luxury                0
Parking               0
Patio/Deck            0
Playground            0
Pool                  0
Refrigerator          0
Storage         

In [10]:
df.shape, y.shape

((79566, 48), (79566,))

In [11]:
print(df["square_feet"].max())

40000


In [28]:
evaluate_df.shape

(19898, 48)

In [29]:
# categorical_cols = df.select_dtypes(include=['object']).columns

# Fit the encoder on the training data
# encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
# encoder.fit(df[["category", "has_photo", "pets_allowed", "source"]])
# x_encoded = encoder.transform(df[["category", "has_photo", "pets_allowed", "source"]])
# # Split back into train and test

# df_one_hot = pd.DataFrame(x_encoded)
# print(df_one_hot.shape)
# df_removed_col = df.drop(columns=["category", "has_photo", "pets_allowed", "source"])
# df_with_one_hot = pd.concat([df_removed_col, df_one_hot], axis=1)

In [30]:
# y_encoded = encoder.transform(evaluate_df[["category", "has_photo", "pets_allowed", "source"]])
# evaluate_df_one_hot = pd.DataFrame(y_encoded)
# evaluate_df_removed_col = evaluate_df.drop(columns=["category", "has_photo", "pets_allowed", "source"])
# evaluate_df_with_one_hot = pd.concat([evaluate_df_removed_col, evaluate_df_one_hot], axis=1)

In [12]:
x_encoded = df.drop(columns=["category", "has_photo", "pets_allowed", "source", "fee"])

In [13]:
x_encoded.shape, y.shape


((79566, 43), (79566,))

In [14]:
x_encoded.info()

<class 'pandas.core.frame.DataFrame'>
Index: 79566 entries, 0 to 79588
Data columns (total 43 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   bathrooms           79566 non-null  float64
 1   bedrooms            79566 non-null  float64
 2   square_feet         79566 non-null  int64  
 3   latitude            79566 non-null  float64
 4   longitude           79566 non-null  float64
 5   time                79566 non-null  int64  
 6   id                  79566 non-null  int64  
 7   max_from_body       79566 non-null  float64
 8   min_from_body       79566 non-null  float64
 9   std_from_body       79566 non-null  float64
 10  mean_from_body      79566 non-null  float64
 11  max_from_title      79566 non-null  float64
 12  min_from_title      79566 non-null  float64
 13  std_from_title      79566 non-null  float64
 14  mean_from_title     79566 non-null  float64
 15  AC                  79566 non-null  int32  
 16  Alarm    

In [15]:
evaluate_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19898 entries, 0 to 19897
Data columns (total 48 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   category            19898 non-null  object 
 1   bathrooms           19898 non-null  float64
 2   bedrooms            19898 non-null  float64
 3   fee                 19898 non-null  object 
 4   has_photo           19898 non-null  object 
 5   pets_allowed        19898 non-null  object 
 6   square_feet         19898 non-null  int64  
 7   latitude            19898 non-null  float64
 8   longitude           19898 non-null  float64
 9   source              19898 non-null  object 
 10  time                19898 non-null  int64  
 11  id                  19898 non-null  int64  
 12  max_from_body       19898 non-null  float64
 13  min_from_body       19898 non-null  float64
 14  std_from_body       19898 non-null  float64
 15  mean_from_body      19898 non-null  float64
 16  max_

In [16]:
X_train, X_test, y_train, y_test = train_test_split(x_encoded, y, test_size=0.2, shuffle=True, random_state=42)

In [17]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [18]:
table = PrettyTable()
table.field_names = ["Model", "MAE"]

dst = DecisionTreeRegressor()
dst.fit(X_train,y_train)
y_pred_dst = dst.predict(X_test)
mae = mean_absolute_error(y_test, y_pred_dst)
table.add_row(["Decision Tree", mae])

lr = LinearRegression(n_jobs = -1)
lr.fit(X_train_scaled,y_train)
y_preds_linear = lr.predict(X_test_scaled)
mae = mean_absolute_error(y_test,y_preds_linear)
table.add_row(["Linear Regressor", mae])

rfr = RandomForestRegressor(n_jobs=-1)
rfr.fit(X_train, y_train)
y_pred = rfr.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
table.add_row(["Random Forest", mae])

hgb = HistGradientBoostingRegressor(max_iter=1000, learning_rate=0.05, max_depth=6)
hgb.fit(X_train, y_train)
y_preds_hgb = hgb.predict(X_test)
mae = mean_absolute_error(y_test, y_preds_hgb)
table.add_row(["Hist Gradient Boost", mae])

xgb = XGBRegressor(learning_rate=0.05, n_estimators=500, max_depth=6,reg_alpha=1, reg_lambda=5)
xgb.fit(X_train,y_train)
y_pred_xgb = xgb.predict(X_test)
mae = mean_absolute_error(y_test, y_pred_xgb)
table.add_row(["XGBoost", mae])

nigga = GradientBoostingRegressor()
nigga.fit(X_train_scaled,y_train)
y_pred_nigga = nigga.predict(X_test_scaled)
mae = mean_absolute_error(y_test, y_pred_nigga)
table.add_row(["Gradient Boost", mae])

negro = ExtraTreesRegressor(n_jobs=-1)
negro.fit(X_train_scaled, y_train)
y_pred_negro = negro.predict(X_test_scaled)
mae = mean_absolute_error(y_test, y_pred_negro)
table.add_row(["Extra Tree", mae])

print(table)

+---------------------+--------------------+
|        Model        |        MAE         |
+---------------------+--------------------+
|    Decision Tree    | 205.14339512379038 |
|   Linear Regressor  | 460.0378613230851  |
|    Random Forest    | 149.62251036822923 |
| Hist Gradient Boost | 219.67711814183832 |
|       XGBoost       | 199.8946836026326  |
|    Gradient Boost   | 275.0247466705827  |
|      Extra Tree     | 144.2048944325751  |
+---------------------+--------------------+


In [19]:
importances = rfr.feature_importances_
feature_names = X_train.columns
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)
print(feat_imp.head(20))  # Show top 20 most important features

longitude          0.304400
square_feet        0.264637
latitude           0.124497
bathrooms          0.050702
bedrooms           0.050067
time               0.045861
id                 0.025809
min_from_body      0.024125
mean_from_body     0.022172
max_from_body      0.014599
std_from_body      0.008715
Gym                0.004666
Pool               0.004400
max_from_title     0.004067
Parking            0.003868
Patio/Deck         0.003148
min_from_title     0.003124
mean_from_title    0.003052
none               0.002839
Elevator           0.002708
dtype: float64


In [20]:
for col in evaluate_df.columns:
    if col not in x_encoded.columns:
        evaluate_df.drop(columns=col, inplace=True)

In [21]:
evaluate_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19898 entries, 0 to 19897
Data columns (total 43 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   bathrooms           19898 non-null  float64
 1   bedrooms            19898 non-null  float64
 2   square_feet         19898 non-null  int64  
 3   latitude            19898 non-null  float64
 4   longitude           19898 non-null  float64
 5   time                19898 non-null  int64  
 6   id                  19898 non-null  int64  
 7   max_from_body       19898 non-null  float64
 8   min_from_body       19898 non-null  float64
 9   std_from_body       19898 non-null  float64
 10  mean_from_body      19898 non-null  float64
 11  max_from_title      19898 non-null  float64
 12  min_from_title      19898 non-null  float64
 13  std_from_title      19898 non-null  float64
 14  mean_from_title     19898 non-null  float64
 15  AC                  19898 non-null  int32  
 16  Alar

In [22]:
evaluate_df.columns == x_encoded.columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True])

In [23]:
# evaluate_df = preprocessing_temp2(evaluate_df)
# display(evaluate_df)
# evaluate_df = evaluate_df.drop(columns=["bedrooms", "bathrooms"])

In [24]:
missing_per_feature = evaluate_df.isnull().sum()
print(missing_per_feature)

bathrooms             0
bedrooms              0
square_feet           0
latitude              0
longitude             0
time                  0
id                    0
max_from_body         0
min_from_body         0
std_from_body         0
mean_from_body        0
max_from_title        0
min_from_title        0
std_from_title        0
mean_from_title       0
AC                    0
Alarm                 0
Basketball            0
Cable or Satellite    0
Clubhouse             0
Dishwasher            0
Doorman               0
Elevator              0
Fireplace             0
Garbage Disposal      0
Gated                 0
Golf                  0
Gym                   0
Hot Tub               0
Internet Access       0
Luxury                0
Parking               0
Patio/Deck            0
Playground            0
Pool                  0
Refrigerator          0
Storage               0
TV                    0
Tennis                0
View                  0
Washer Dryer          0
Wood Floors     

In [25]:
for i in range (len(evaluate_df)):
    if (pd.isnull(evaluate_df.iloc[i]["bathrooms"])):
        print(evaluate_df.iloc[i])

In [26]:
rfr_online = RandomForestRegressor(n_jobs=-1)
rfr_online.fit(x_encoded, y)

# Now you can safely predict
y_pred = rfr_online.predict(evaluate_df)
results_df = pd.DataFrame({
    "Id": evaluate_df["id"],
    "Predicted": y_pred.astype(float)  # or round if you want integers
})

# Save to CSV
results_df.to_csv("predictions.csv", index=False)

In [20]:
hgb_online = HistGradientBoostingRegressor(max_iter=1000, learning_rate=0.05, max_depth=6)
hgb_online.fit(x_encoded,y)
y_pred = hgb_online.predict(evaluate_df)
results_df = pd.DataFrame({
    "Id": evaluate_df["id"],
    "Predicted": y_pred.astype(float)  # or round if you want integers
})

# Save to CSV
results_df.to_csv("predictions.csv", index=False)

In [21]:
xgb_online = XGBRegressor()
xgb_online.fit(x_encoded,y)
y_pred = xgb_online.predict(evaluate_df)
results_df = pd.DataFrame({
    "Id": evaluate_df["id"],
    "Predicted": y_pred.astype(float)  # or round if you want integers
})

# Save to CSV
results_df.to_csv("predictions.csv", index=False)

In [22]:
missing_per_feature = evaluate_df.isnull().sum()
print(missing_per_feature)

bathrooms               1
bedrooms                0
square_feet             0
latitude                0
longitude               0
time                    0
id                      0
square_feet_quantile    1
max_from_body           0
min_from_body           0
std_from_body           0
mean_from_body          0
max_from_title          0
min_from_title          0
std_from_title          0
mean_from_title         0
AC                      0
Alarm                   0
Basketball              0
Cable or Satellite      0
Clubhouse               0
Dishwasher              0
Doorman                 0
Elevator                0
Fireplace               0
Garbage Disposal        0
Gated                   0
Golf                    0
Gym                     0
Hot Tub                 0
Internet Access         0
Luxury                  0
Parking                 0
Patio/Deck              0
Playground              0
Pool                    0
Refrigerator            0
Storage                 0
TV          

In [27]:
extra_tree = ExtraTreesRegressor(n_jobs=-1)
extra_tree.fit(x_encoded, y)
y_pred = extra_tree.predict(evaluate_df)
results_df = pd.DataFrame({
    "Id": evaluate_df["id"],
    "Predicted": y_pred.astype(float)  # or round if you want integers
})

# Save to CSV
results_df.to_csv("predictions.csv", index=False)

In [ ]:
importances = rfr_online.feature_importances_
feature_names = X_train.columns
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)
print(feat_imp.head(50))  # Show top 20 most important features

In [ ]:
#api_key: c97d8bd8e2dcfa84c1e1c72140d1582f23e136d1ba0fd19d8a39915e7db6207e